# Pruebas A/B: código seleccionado

Bitácora IE12_2 asignada a Keber. El dataset marketing_AB.csv no está incluido. La decisión de excluir usuarios de alta exposición y la lectura del desbalance entre grupos necesitan revisión antes de citar conclusiones. El escenario CORFO es sintético.


## Exploración de valores extremos


In [ ]:
# Tu código aquí ...
# Calcular los cuantiles y el rango intercuartílico (IQR)
Q1 = ad_campaign['total_ads'].quantile(0.25)  # Primer cuartil (25%)
Q3 = ad_campaign['total_ads'].quantile(0.75)  # Tercer cuartil (75%)
IQR = Q3 - Q1  # Rango intercuartílico

# Calcular los límites inferior y superior para identificar outliers
down_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Identificar outliers
outliers = ad_campaign[(ad_campaign['total_ads'] < down_limit) | (ad_campaign['total_ads'] > upper_limit)]

# Crear el dataset sin outliers
no_outliers_df = ad_campaign[(ad_campaign['total_ads'] >= down_limit) & (ad_campaign['total_ads'] <= upper_limit)]

# Calcular tasas de conversión
conversion_total = ad_campaign['converted'].mean()
conversion_no_outliers = no_outliers_df['converted'].mean()

# Diferencia relativa entre las tasas
diferencia_relativa = (conversion_no_outliers - conversion_total) / conversion_total * 100

# Tasa de conversión entre los outliers por grupo
conversion_outliers = outliers.groupby('test_group')['converted'].mean()

# Porcentaje de usuarios que son outliers por grupo
outliers_percentage = outliers.groupby('test_group').size() / ad_campaign.groupby('test_group').size() * 100

print(f"1.a Tasa de conversión general utilizando todo el dataset: {conversion_total:.4f}")
print(f"1.b Tasa de conversión utilizando el dataset limpio (sin outliers): {conversion_no_outliers:.4f}")
print(f"2. Diferencia relativa entre ambas tasas: {diferencia_relativa:.2f}%")
print(f"3.a Tasa de conversión entre los outliers:\n{conversion_outliers}")
print(f"3.b Porcentaje de usuarios que son outliers:\n{outliers_percentage}")


## Intervalo de confianza Wald


In [ ]:
# Calcular el intervalo de confianza (IC) usando el método Wald
from statsmodels.stats.proportion import confint_proportions_2indep

ci = confint_proportions_2indep(
    count1=successes[0],  # éxitos en el grupo de anuncios (tratamiento)
    nobs1=nobs[0],        # total de usuarios en el grupo de anuncios
    count2=successes[1],  # éxitos en el grupo de PSA (control)
    nobs2=nobs[1],        # total de usuarios en el grupo de PSA
    alpha=0.05,           # para un IC del 95%
    method='wald'         # método Wald para el intervalo de confianza
)

print(f'IC del 95% para la diferencia de proporciones: ({ci[0]:.2%} - {ci[1]:.2%})')
print(f'\nDiferencia actual en la tasa de conversión: {diff:.2%}')

## Submuestreo y nueva prueba


In [ ]:
# Tu código aquí ...
# Submuestreo del grupo de tratamiento
np.random.seed(42)  # Para reproducibilidad
control_size = len(control_group)  # Tamaño del grupo de control
treatment_sample = treatment_group.sample(n=control_size, random_state=42)

# Prueba Z para dos proporciones con los datos ajustados
# Contar éxitos y tamaños de muestra
successes_adjusted = [
    sum(treatment_sample['converted'] == True),  # Éxitos en el grupo de tratamiento ajustado
    sum(control_group['converted'] == True)     # Éxitos en el grupo de control
]

nobs_adjusted = [
    len(treatment_sample),  # Tamaño del grupo de tratamiento ajustado
    len(control_group)      # Tamaño del grupo de control
]

# Prueba Z
z_stat_adjusted, p_value_adjusted = proportions_ztest(
    count=successes_adjusted,
    nobs=nobs_adjusted,
    alternative='larger'
)

# Intervalo de confianza para los datos ajustados
ci_adjusted = confint_proportions_2indep(
    count1=successes_adjusted[0],
    nobs1=nobs_adjusted[0],
    count2=successes_adjusted[1],
    nobs2=nobs_adjusted[1],
    alpha=0.05
)

# Comparar con los resultados antes del ajuste (desbalanceado)
# Prueba Z para los datos desbalanceados
z_stat_original, p_value_original = proportions_ztest(
    count=successes,
    nobs=nobs,
    alternative='larger'
)

# Intervalo de confianza para los datos desbalanceados
ci_original = confint_proportions_2indep(
    count1=successes[0],
    nobs1=nobs[0],
    count2=successes[1],
    nobs2=nobs[1],
    alpha=0.05,
    method='wald'
)

# Resultados
{
    "Antes del ajuste": {
        "Estadístico Z": f"{z_stat_original:.2f}",
        "p-value": p_value_original,
        "IC (95%)": f"{ci_original[0]*100:.2f}% - {ci_original[1]*100:.2f}%"
    },
    "Después del ajuste": {
        "Estadístico Z": f"{z_stat_adjusted:.2f}",
        "p-value": p_value_adjusted,
        "IC (95%)": f"{ci_adjusted[0]*100:.2f}% - {ci_adjusted[1]*100:.2f}%"
    }
}

## Escenario sintético CORFO


In [ ]:
# Tu código aquí ...
from statsmodels.stats.power import NormalIndPower

# Paso 1: Formular las hipótesis
hipotesis = """
H0: La tasa de inscripción para el Mensaje B (tratamiento) es menor o igual que la del Mensaje A (control).
H1: La tasa de inscripción para el Mensaje B (tratamiento) es mayor que la del Mensaje A (control).
"""

# Paso 2: Simulación de datos
np.random.seed(42)  # Para reproducibilidad

# Tamaño de muestra y tasas de conversión
n_control = 1000
n_treatment = 1000
p_control = 0.03  # Tasa de conversión para Mensaje A
p_treatment = 0.05  # Tasa de conversión para Mensaje B

# Generar datos simulados usando distribuciones binomiales
control_data = np.random.binomial(1, p_control, n_control)
treatment_data = np.random.binomial(1, p_treatment, n_treatment)

# Contar éxitos (inscripciones) en cada grupo
successes = [sum(treatment_data), sum(control_data)]
nobs = [n_treatment, n_control]

# Paso 3: Prueba Z para dos proporciones
z_stat, p_value = proportions_ztest(count=successes, nobs=nobs, alternative='larger')

# Intervalo de confianza del 95% para la diferencia de proporciones
ci = confint_proportions_2indep(
    count1=successes[0],
    nobs1=nobs[0],
    count2=successes[1],
    nobs2=nobs[1],
    alpha=0.05,
    method='wald'
)

# Paso 4: Cálculo del tamaño del efecto (Cohen’s h)
def cohen_h(p1, p2):
    return 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))

p_treatment_actual = successes[0] / nobs[0]
p_control_actual = successes[1] / nobs[1]
effect_size = cohen_h(p_treatment_actual, p_control_actual)

# Paso 5: Análisis de poder
power_analysis = NormalIndPower()
power = power_analysis.solve_power(
    effect_size=effect_size,
    nobs1=nobs[0],
    alpha=0.05,
    ratio=nobs[1] / nobs[0],
    alternative='larger'
)

# Resultados
results = {
    "Hipótesis": hipotesis,
    "Éxitos (tratamiento)": successes[0],
    "Éxitos (control)": successes[1],
    "Tasa de inscripción (tratamiento)": f"{p_treatment_actual:.2%}",
    "Tasa de inscripción (control)": f"{p_control_actual:.2%}",
    "Diferencia de tasas": f"{(p_treatment_actual - p_control_actual):.2%}",
    "Estadístico Z": z_stat,
    "Valor p": p_value,
    "IC (95%) para la diferencia de proporciones": ci,
    "Tamaño del efecto (Cohen's h)": effect_size,
    "Poder estadístico": power,
}

results
